# Approach 1: Fused Embeddings (Baseline)

This notebook implements the simplest approach from the whitepaper: fusing visual, audio, and transcription embeddings into a single vector per segment.

**Formula:** `E_fused = normalize(w_v * E_visual + w_a * E_audio + w_t * E_transcript)`

**Default weights:** visual=0.8, audio=0.1, transcription=0.05

In [ ]:
import json
import numpy as np
import boto3
from botocore.config import Config

# Auto-load configuration from 01_setup_and_embedding.ipynb
with open("config.json") as f:
    config = json.load(f)

REGION = config["REGION"]
S3_BUCKET = config["S3_BUCKET"]
MODEL_ID_SYNC = config["MODEL_ID_SYNC"]
VECTOR_BUCKET = config["VECTOR_BUCKET"]
DATA_S3_PREFIX = config["DATA_S3_PREFIX"]
RESULTS_S3_PREFIX = config["RESULTS_S3_PREFIX"]

print(f"✅ Configuration loaded from config.json")
print(f"   S3 Bucket: {S3_BUCKET}")
print(f"   Vector Bucket: {VECTOR_BUCKET}")

bedrock = boto3.client("bedrock-runtime", region_name=REGION, config=Config(signature_version='v4'))
s3 = boto3.client("s3", region_name=REGION)
s3v = boto3.client("s3vectors", region_name=REGION)

def fmt_time(sec):
    """Convert seconds to mm:ss format"""
    m, s = divmod(sec, 60)
    return f"{int(m):02d}:{s:04.1f}"

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def embed_text(query):
    response = bedrock.invoke_model(
        modelId=MODEL_ID_SYNC,
        body=json.dumps({"inputType": "text", "text": {"inputText": query}})
    )
    result = json.loads(response["body"].read())
    return result["data"][0]["embedding"]

print("Setup complete")

## 1. Load Embeddings

In [2]:
# Load embeddings from S3
def load_embeddings_from_s3(modality):
    s3_key = f"{DATA_S3_PREFIX}embeddings_{modality}.json"
    obj = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
    return json.loads(obj["Body"].read().decode("utf-8"))

visual = load_embeddings_from_s3("visual")
audio = load_embeddings_from_s3("audio")
transcription = load_embeddings_from_s3("transcription")

print(f"Visual segments: {len(visual)}")
print(f"Audio segments: {len(audio)}")
print(f"Transcription segments: {len(transcription)}")

Visual segments: 161
Audio segments: 161
Transcription segments: 161


## 2. Build Fused Index

Align segments by time range and compute fused embeddings.

In [3]:
# Fusion weights (from whitepaper Section 2.2)
W_V, W_A, W_T = 0.8, 0.1, 0.05

def fuse_embeddings(v_emb, a_emb, t_emb, w_v=W_V, w_a=W_A, w_t=W_T):
    """Fuse 3 modality embeddings into a single vector (Equation 1 from whitepaper)"""
    v = np.array(v_emb)
    a = np.array(a_emb) if a_emb is not None else np.zeros_like(v)
    t = np.array(t_emb) if t_emb is not None else np.zeros_like(v)
    
    fused = w_v * v + w_a * a + w_t * t
    norm = np.linalg.norm(fused)
    if norm > 0:
        fused = fused / norm
    return fused.tolist()

# Build time-aligned index
# Create lookup by (startSec, endSec) for each modality
def build_time_lookup(embeddings):
    lookup = {}
    for e in embeddings:
        key = (round(e["startSec"], 1), round(e["endSec"], 1))
        lookup[key] = e["embedding"]
    return lookup

visual_lookup = build_time_lookup(visual)
audio_lookup = build_time_lookup(audio)
trans_lookup = build_time_lookup(transcription)

# Get all unique time segments from visual (primary modality)
fused_index = []
for seg in visual:
    key = (round(seg["startSec"], 1), round(seg["endSec"], 1))
    v_emb = seg["embedding"]
    a_emb = audio_lookup.get(key)
    t_emb = trans_lookup.get(key)
    
    fused = fuse_embeddings(v_emb, a_emb, t_emb)
    fused_index.append({
        "embedding": fused,
        "startSec": seg["startSec"],
        "endSec": seg["endSec"],
        "has_audio": a_emb is not None,
        "has_transcription": t_emb is not None
    })

print(f"Fused index: {len(fused_index)} segments")
print(f"  Dimension: {len(fused_index[0]['embedding'])}")
coverage = sum(1 for s in fused_index if s["has_audio"] and s["has_transcription"])
print(f"  Full 3-modality coverage: {coverage}/{len(fused_index)}")

Fused index: 161 segments
  Dimension: 512
  Full 3-modality coverage: 161/161


## 2.5 Store Fused Embeddings in S3 Vectors

In [4]:
FUSED_INDEX_NAME = "fused"

# Create fused index (skip if exists)
try:
    s3v.get_index(vectorBucketName=VECTOR_BUCKET, indexName=FUSED_INDEX_NAME)
    print(f"Index '{FUSED_INDEX_NAME}' already exists")
except s3v.exceptions.NotFoundException:
    s3v.create_index(
        vectorBucketName=VECTOR_BUCKET,
        indexName=FUSED_INDEX_NAME,
        dataType="float32",
        dimension=512,
        distanceMetric="cosine"
    )
    print(f"Created index '{FUSED_INDEX_NAME}' (512-dim, cosine)")

# Store fused embeddings
BATCH_SIZE = 50
total = 0
for i in range(0, len(fused_index), BATCH_SIZE):
    batch = fused_index[i:i+BATCH_SIZE]
    vectors = []
    for seg in batch:
        vectors.append({
            "key": f"fused_{seg['startSec']:.1f}_{seg['endSec']:.1f}",
            "data": {"float32": [float(x) for x in seg["embedding"]]},
            "metadata": {
                "startSec": seg["startSec"],
                "endSec": seg["endSec"]
            }
        })
    s3v.put_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=FUSED_INDEX_NAME,
        vectors=vectors
    )
    total += len(vectors)

print(f"Stored {total} fused vectors in S3 Vectors")

Index 'fused' already exists
Stored 161 fused vectors in S3 Vectors


## 3. Search with Fused Embeddings

Test with queries spanning different modality intents.

In [5]:
TEST_QUERIES = [
    {"query": "a player kicking the ball towards the goal", "intent": "visual"},
    {"query": "crowd cheering and celebrating loudly", "intent": "audio"},
    {"query": "the commentator describes the player's performance", "intent": "transcription"},
    {"query": "a goal being scored with the stadium erupting", "intent": "visual+audio"},
    {"query": "the announcer explaining a foul while the crowd boos", "intent": "mixed"},
    {"query": "players running on the soccer field", "intent": "visual"},
    {"query": "referee whistle blowing during the match", "intent": "audio"},
    {"query": "the commentator discusses the team's strategy", "intent": "transcription"},
]

def search_fused_s3v(query_text, top_k=5):
    """Search the fused index in S3 Vectors using ANN"""
    q_emb = embed_text(query_text)
    
    response = s3v.query_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=FUSED_INDEX_NAME,
        topK=top_k,
        queryVector={"float32": [float(x) for x in q_emb]},
        returnMetadata=True,
        returnDistance=True
    )
    
    results = []
    for v in response["vectors"]:
        results.append({
            "startSec": v["metadata"]["startSec"],
            "endSec": v["metadata"]["endSec"],
            "similarity": v["distance"]
        })
    return results

# Run searches using S3 Vectors
fused_results = {}
for item in TEST_QUERIES:
    query = item["query"]
    intent = item["intent"]
    print(f"\nQuery: \"{query}\" (intent: {intent})")
    print("-" * 60)
    
    results = search_fused_s3v(query)
    fused_results[query] = results
    
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{fmt_time(r['startSec'])} - {fmt_time(r['endSec'])}] distance: {r['similarity']:.4f}")


Query: "a player kicking the ball towards the goal" (intent: visual)
------------------------------------------------------------
  1. [16:27.8 - 16:34.2] distance: 0.8812
  2. [13:08.0 - 13:18.0] distance: 0.8831
  3. [15:22.8 - 15:28.2] distance: 0.8852
  4. [02:04.0 - 02:10.2] distance: 0.8857
  5. [07:40.0 - 07:45.5] distance: 0.8868

Query: "crowd cheering and celebrating loudly" (intent: audio)
------------------------------------------------------------
  1. [14:51.2 - 14:55.8] distance: 0.8893
  2. [10:27.0 - 10:33.2] distance: 0.8963
  3. [09:38.0 - 09:42.0] distance: 0.9015
  4. [14:55.8 - 15:03.0] distance: 0.9221

Query: "the commentator describes the player's performance" (intent: transcription)
------------------------------------------------------------
  1. [00:05.8 - 00:11.5] distance: 0.8972
  2. [00:22.8 - 00:28.0] distance: 0.9010
  3. [08:58.2 - 09:04.0] distance: 0.9064
  4. [00:17.2 - 00:22.8] distance: 0.9077
  5. [03:43.2 - 03:49.8] distance: 0.9079

Query: "a

## 4. Limitations of Fused Embeddings

The fused approach uses the **same weights for every query**, regardless of intent:
- A visual query gets the same 80% visual bias as an audio query
- You can't tell which modality contributed to a match
- Changing weights requires re-computing all fused embeddings (irreversible)

In [6]:
print("Limitation: 예시 가중치 기반으로 쿼리를 진행했을 떄,")
print(f"  Visual weight:        {W_V}")
print(f"  Audio weight:         {W_A}")
print(f"  Transcription weight: {W_T}")
print()
print("This means:")
print('  - "crowd cheering and celebrating loudly" 80%는 visual에서 결과를 가져오려 함.')
print('  - 어디서 매칭됐는지 비교 불가하고 조정이 어려움')


Limitation: 예시 가중치 기반으로 쿼리를 진행했을 떄,
  Visual weight:        0.8
  Audio weight:         0.1
  Transcription weight: 0.05

This means:
  - "crowd cheering and celebrating loudly" 80%는 visual에서 결과를 가져오려 함.
  - 어디서 매칭됐는지 비교 불가하고 조정이 어려움


In [7]:
# Save results to S3
results_data = {
    "approach": "fused_embeddings",
    "weights": {"visual": W_V, "audio": W_A, "transcription": W_T},
    "results": {q: r for q, r in fused_results.items()}
}

s3_key = f"{RESULTS_S3_PREFIX}fused_results.json"
s3.put_object(
    Bucket=S3_BUCKET,
    Key=s3_key,
    Body=json.dumps(results_data, indent=2),
    ContentType="application/json"
)

print(f"Results saved to s3://{S3_BUCKET}/{s3_key}")

Results saved to s3://twelvelabs-marengo-test-1755757894/output/marengo-multivector/results/fused_results.json


---

## 정리: Fused Embeddings 접근법

### 구조
```
[영상] → Marengo 3.0 → visual 임베딩 (512-dim)
                       → audio 임베딩 (512-dim)      → 가중합 → fused 임베딩 (512-dim) → S3 Vectors 저장
                       → transcription 임베딩 (512-dim)

[검색 쿼리] → Marengo text embed → fused 인덱스에서 cosine similarity 검색
```

### 핵심 공식
```
E_fused = normalize(0.8 × E_visual + 0.1 × E_audio + 0.05 × E_transcription)
```

### 이 노트북의 목적
- **Baseline으로서의 한계를 보여주는 것**이 핵심
- 3개 모달리티 임베딩을 저장 시점에 하나로 합쳐버리는 가장 단순한 방식
- 합칠 때 사용한 가중치(0.8/0.1/0.05)가 **모든 쿼리에 동일하게 적용됨**

### 확인하려는 것
| 쿼리 intent | 기대 | 실제 (fused의 문제) |
|---|---|---|
| visual | visual 기반 검색 → 적절한 결과 | O - 80%가 visual이므로 잘 동작 |
| audio | audio 기반 검색 → 함성/휘슬 구간 | X - 80%가 visual이라 audio 정보가 묻힘 |
| transcription | 해설 내용 기반 검색 | X - transcription 비중 5%라 거의 무시됨 |

### 한계
1. **비가역적**: 한번 합치면 원본 모달리티 임베딩으로 되돌릴 수 없음
2. **쿼리 무관**: "crowd cheering"이든 "players running"이든 항상 같은 가중치
3. **디버깅 불가**: 검색 결과에서 어떤 모달리티가 기여했는지 알 수 없음
4. **가중치 변경 = 전체 재처리**: 가중치를 바꾸려면 모든 영상의 fused 임베딩을 다시 계산해야 함

### 다음 단계
→ 03번에서 임베딩을 합치지 않고 따로 두면 어떤 장점이 있는지 확인